In [8]:
# Breast Histopathology image

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

count = 0

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
        count += 1
        if count == 3:
            break
    if count == 3:
        break

/kaggle/input/datasets/samishirzad/444444/breast_dataset/val/benign/10254_idx5_x501_y1851_class0.png
/kaggle/input/datasets/samishirzad/444444/breast_dataset/val/benign/10253_idx5_x2051_y1001_class0.png
/kaggle/input/datasets/samishirzad/444444/breast_dataset/val/benign/10254_idx5_x951_y1801_class0.png


In [9]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

# GÜNCEL VERİ SETİ YOLU
DATA_DIR = "/kaggle/input/datasets/samishirzad/444444/breast_dataset"
TRAIN_DIR = DATA_DIR + "/train"
VAL_DIR   = DATA_DIR + "/val"
TEST_DIR  = DATA_DIR + "/test"
IMG_SIZE = (224,224)
BATCH_SIZE = 16

# Generators
train_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=15, zoom_range=0.15,
    horizontal_flip=True, vertical_flip=True, brightness_range=[0.8, 1.2]
)
val_test_gen = ImageDataGenerator(rescale=1./255)

train_data_m1 = train_gen.flow_from_directory(TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="binary")
val_data_m1   = val_test_gen.flow_from_directory(VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="binary")
test_data_m1  = val_test_gen.flow_from_directory(TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE, class_mode="binary", shuffle=False)

# MODEL-1 Architecture
base_model1 = MobileNetV2(input_shape=(224,224,3), include_top=False, weights="imagenet")
for layer in base_model1.layers[:-40]:
    layer.trainable = False
for layer in base_model1.layers[-40:]:
    layer.trainable = True

x = GlobalAveragePooling2D()(base_model1.output)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output1 = Dense(1, activation="sigmoid")(x)
model1 = Model(base_model1.input, output1)

# Cosine Decay Setup 
steps_per_epoch_m1 = len(train_data_m1)
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-4,
    decay_steps=steps_per_epoch_m1*40,
    alpha=1e-2
)
model1.compile(optimizer=Adam(lr_schedule), loss="binary_crossentropy", metrics=["accuracy"])

early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)
class_weight_dict = {0: 3, 1: 1}

print("MODEL-1 Training is starting...")
history1 = model1.fit(
    train_data_m1, 
    validation_data=val_data_m1, 
    epochs=15, 
    callbacks=[early_stop], 
    class_weight=class_weight_dict
)

# TEST Predictions (Clean Data)
y_true = test_data_m1.classes
y_prob1 = model1.predict(test_data_m1).flatten()
y_pred1 = (y_prob1 > 0.40).astype(int)
print("MODEL-1 CLASSIFICATION REPORT (CLEAN DATA)")
print(classification_report(y_true, y_pred1, target_names=["Benign", "Malignant"]))

Found 7000 images belonging to 2 classes.
Found 1500 images belonging to 2 classes.
Found 1500 images belonging to 2 classes.
MODEL-1 Training is starting...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 125s 246ms/step - accuracy: 0.7316 - loss: 0.8021 - val_accuracy: 0.8047 - val_loss: 0.4256
Epoch 2/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 91s 209ms/step - accuracy: 0.7948 - loss: 0.6372 - val_accuracy: 0.5060 - val_loss: 1.3258
Epoch 3/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 89s 204ms/step - accuracy: 0.8162 - loss: 0.5708 - val_accuracy: 0.6000 - val_loss: 0.7284
Epoch 4/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 90s 204ms/step - accuracy: 0.8440 - loss: 0.5088 - val_accuracy: 0.8053 - val_loss: 0.4732
Epoch 5/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 94s 215ms/step - accuracy: 0.8420 - loss: 0.5033 - val_accuracy: 0.5047 - val_loss: 1.9128
Epoch 6/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 90s 205ms/step - accuracy: 0.8508 - loss: 0.4790 - val_accuracy: 0.7620 - val_loss: 0.4972
Epoch 7/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 89s 202ms/step - accuracy: 0.8641 - loss: 0.4447 - val_accuracy: 0.6947 - val_loss: 0.6550
Epoch 8/15
438/438 ━━━━━━━━━━━━━━━━━━━━ 88s 202ms/step - accuracy: 0.8618 - loss: 

In [10]:
# ROBUSTNESS ANALYSIS; NOISE INJECTION & FILTERING EVALUATION
# Objective: Evaluate model performance under Gaussian noise and various spatial filters.
import cv2
import numpy as np
from skimage.util import random_noise
from sklearn.metrics import accuracy_score

print(" " + "="*65)
print("INITIATING ROBUSTNESS AND FILTERING EVALUATION")
print("="*65)

# Extract file paths and ground truth labels from the test generator
image_paths = test_data_m1.filepaths
y_true = test_data_m1.classes

print("Checking baseline accuracy with OpenCV pipeline...")
clean_images = []
for path in image_paths:
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))
    clean_images.append(img.astype(np.float32) / 255.0)

clean_images = np.array(clean_images)
clean_preds = model1.predict(clean_images, batch_size=32, verbose=0)
clean_acc = accuracy_score(y_true, (clean_preds > 0.40).astype(int).flatten())
print(f"CLEAN DATA ACCURACY: {clean_acc*100:.2f}%")

# Define Noise Injection and Filtering Functions
def add_gaussian_noise(img, variance, seed_val):
    """Injects Gaussian noise with a specified variance and fixed seed."""
    
    np.random.seed(seed_val) 
    noisy_img = random_noise(img, mode='gaussian', mean=0, var=variance)
    noisy_img = np.clip(noisy_img * 255, 0, 255).astype(np.uint8)
    return noisy_img

def apply_filter(img, filter_type):
    """Applies the selected spatial filter to the noisy image."""
    if filter_type == "No Filter":
        return img
    elif filter_type == "Median 3x3":
        return cv2.medianBlur(img, 3)
    elif filter_type == "Median 5x5":
        return cv2.medianBlur(img, 5)
    elif filter_type == "Median 7x7":
        return cv2.medianBlur(img, 7)
    elif filter_type == "Median 9x9":
        return cv2.medianBlur(img, 9)
    elif filter_type == "Median 11x11":
        return cv2.medianBlur(img, 11)
    elif filter_type == "Standard Average":
        return cv2.blur(img, (3, 3))
    elif filter_type == "Weighted Average":
        kernel = np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], dtype=np.float32) / 16.0
        return cv2.filter2D(img, -1, kernel)

# Experimental Setup: Noise Variances and Filter Types
noise_levels = {
    "Low (0.005)": 0.005, 
    "Medium (0.02)": 0.02, 
    "High (0.05)": 0.05
}

filter_types = [
    "No Filter", "Median 3x3", "Median 5x5", "Median 7x7", 
    "Median 9x9", "Median 11x11", "Standard Average", "Weighted Average"
]

# Evaluation Loop
for noise_name, variance in noise_levels.items():
    print(f">>> NOISE LEVEL: {noise_name} <<<")
    print("-" * 45)
    
    for f_type in filter_types:
        processed_images = []
        
        # We process all test images; process noise, apply filters, and normalize.
        for i, path in enumerate(image_paths):
            img = cv2.imread(path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            
            
            noisy_img = add_gaussian_noise(img, variance, seed_val=42+i)
            filtered_img = apply_filter(noisy_img, f_type)
            
            input_img = filtered_img.astype(np.float32) / 255.0
            processed_images.append(input_img)
            
        # Convert to numpy array and perform batch inference
        processed_images = np.array(processed_images)
        predictions = model1.predict(processed_images, batch_size=32, verbose=0)
        
        # Calculate Accuracy (Threshold = 0.40)
        y_pred = (predictions > 0.40).astype(int).flatten()
        acc = accuracy_score(y_true, y_pred)
        
        # Display Results
        print(f"{f_type:<20} | Accuracy: {acc*100:.2f}%")

print("EVALUATION COMPLETED SUCCESSFULLY.")

INITIATING ROBUSTNESS AND FILTERING EVALUATION
Checking baseline accuracy with OpenCV pipeline...
CLEAN DATA ACCURACY: 79.20%
>>> NOISE LEVEL: Low (0.005) <<<
---------------------------------------------
No Filter            | Accuracy: 51.73%
Median 3x3           | Accuracy: 76.60%
Median 5x5           | Accuracy: 80.53%
Median 7x7           | Accuracy: 81.20%
Median 9x9           | Accuracy: 82.13%
Median 11x11         | Accuracy: 80.00%
Standard Average     | Accuracy: 78.20%
Weighted Average     | Accuracy: 70.53%
>>> NOISE LEVEL: Medium (0.02) <<<
---------------------------------------------
No Filter            | Accuracy: 50.00%
Median 3x3           | Accuracy: 53.87%
Median 5x5           | Accuracy: 66.00%
Median 7x7           | Accuracy: 74.87%
Median 9x9           | Accuracy: 77.20%
Median 11x11         | Accuracy: 76.53%
Standard Average     | Accuracy: 55.00%
Weighted Average     | Accuracy: 51.00%
>>> NOISE LEVEL: High (0.05) <<<
-----------------------------------------